# **Modelos LSTM — Frontera 01 La Aurora**

En este notebook se modela con redes **LSTM** la serie mensual de viajeros que ingresan por la frontera **01 La Aurora**, que es el aeropuerto internacional y el punto de ingreso con mayor flujo de todos los analizados en el Laboratorio 1. Se construyen varias configuraciones de LSTM, se tunean sus hiperparámetros y se usa la mejor para predecir, para poder finalmente contrastar su desempeño contra el mejor modelo que se obtuvo para esta serie en el laboratorio anterior.

Cabe mencionar que esta serie es la de mayor estacionalidad de todas las trabajadas, con una autocorrelación de 0.77 en el mes 12, lo cual tiene sentido dado que el turismo aéreo se concentra en temporadas específicas del año, a diferencia de las fronteras terrestres que reciben un flujo más constante. Bajo esta idea, resulta interesante ver si una LSTM logra descubrir ese ciclo anual por sí sola, ya que a diferencia del SARIMA aquí no se le indica la estacionalidad de forma explícita.

## **Conjuntos de entrenamiento y prueba**

Para poder comparar los modelos LSTM contra los que ya se construyeron, se trabaja con **los mismos
conjuntos de entrenamiento y prueba del Laboratorio 1**, sin volver a decidir nada sobre los datos.
Es decir, se parte de los mismos `entrenamiento.csv` y `prueba.csv`, que ya vienen filtrados a
Turista + Excursionista y particionados de forma temporal 70/30, de tal forma que el entrenamiento
cubre de enero de 2009 a marzo de 2021 (147 meses) y la prueba de abril de 2021 a junio de 2026
(63 meses). Cabe mencionar que la partición respeta el orden cronológico y no es aleatoria, ya que
en una serie de tiempo ese orden es justamente lo que se quiere modelar.

Bajo esta idea, la serie de **01 La Aurora** se reconstruye filtrando la columna `Frontera` por ese valor y sumando los viajeros de cada mes, y se le aplica la misma
transformación logarítmica que se justificó en el laboratorio anterior, dado que el lambda de
Box-Cox sobre el tramo sin pandemia (2009-2019) resultó en -0.464, es decir cercano a 0. Adicional,
las métricas se calculan igual que antes, deshaciendo el logaritmo con la exponencial para reportar
el MAE y el RMSE en número de viajeros, de tal forma que los resultados sean comparables uno a uno
contra el SARIMA, Holt-Winters, el suavizamiento exponencial simple, el seasonal naive y Prophet.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# El codigo compartido vive en src/ (construccion de las series, metricas y utilidades de LSTM).
RAIZ = Path.cwd().parents[1] if Path.cwd().name == "lstm" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import config
import utils

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 5)

SERIE = "fro_aurora"
PERIODO = config.PERIODO

# Las series se reconstruyen desde los CSV del Laboratorio 1, con la misma agrupacion mensual.
serie_train = utils.construir_serie(SERIE, "train")
serie_prueba = utils.construir_serie(SERIE, "prueba")

# Transformacion logaritmica, la misma que se eligio en el Laboratorio 1 para esta serie.
serie_train_log = np.log(serie_train)
serie_prueba_log = np.log(serie_prueba)

print("Serie:", config.SERIES_LAB1[SERIE]["etiqueta"])
print(f"Entrenamiento: {serie_train.index.min():%Y-%m} a {serie_train.index.max():%Y-%m}  "
      f"({len(serie_train)} meses)")
print(f"Prueba       : {serie_prueba.index.min():%Y-%m} a {serie_prueba.index.max():%Y-%m}  "
      f"({len(serie_prueba)} meses)")
print("Frecuencia   :", serie_train.index.freqstr, "(mensual, inicio de mes)")
print("Meses sin registro:", int(serie_train.isna().sum() + serie_prueba.isna().sum()))
print()
print("Mejor modelo del Laboratorio 1 para esta serie:")
print(f"  {config.MEJORES_LAB1[SERIE]['modelo']}  "
      f"MAE={config.MEJORES_LAB1[SERIE]['MAE']:,.0f}  "
      f"RMSE={config.MEJORES_LAB1[SERIE]['RMSE']:,.0f}")

serie_train.tail()


Serie: 01 La Aurora
Entrenamiento: 2009-01 a 2021-03  (147 meses)
Prueba       : 2021-04 a 2026-06  (63 meses)
Frecuencia   : MS (mensual, inicio de mes)
Meses sin registro: 0

Mejor modelo del Laboratorio 1 para esta serie:
  Suav. exp. simple  MAE=36,822  RMSE=42,053


Fecha
2020-11-01    38988.000000
2020-12-01    50370.000000
2021-01-01    40801.000000
2021-02-01    28780.000000
2021-03-01    58806.139619
Freq: MS, Name: Viajero, dtype: float64

## **1.2. Modelos LSTM y tuneo de parámetros**

Antes de entrenar conviene dejar claro qué se está tuneando y por qué, ya que en una LSTM hay
muchísimo que se puede mover y no todo pesa lo mismo. 

| Eje | Valores | Por qué se prueba |
|---|---|---|
| **Presentación** | nivel · diferenciada | Es el eje que más puede pesar, ya que no cambia qué tan bien la red resuelve el problema sino cuál es el problema. Con nivel la red recibe log(viajeros) y aprende a predecir el valor absoluto, pero las redes extrapolan mal fuera del rango que vieron entrenando, de tal forma que tiende a aplanarse. Con diferenciada recibe los cambios mes a mes, que es un objetivo estacionario, y el nivel se reconstruye acumulando; puede despegar, aunque a cambio los errores se van acumulando. |
| **Ventana** | 12 / 24 | Es cuántos meses ve la red antes de predecir. Con 12 alcanza a ver un ciclo anual completo y con 24 puede comparar un año contra el anterior. Cabe mencionar que por debajo de 12 la red no llegaría a ver el ciclo, de tal forma que no podría aprender la estacionalidad ni queriendo. |
| **Arquitectura** | A simple (16/32/64) · B apilada (32/64) · C bidireccional (32) | Son las tres configuraciones distintas. La **A** es una sola capa LSTM y es la línea base. La **B** apila dos capas con dropout de 0.2, es decir más capacidad, aunque con 147 observaciones el riesgo de sobreajuste es real y el dropout está justamente para contenerlo. La **C** recorre la ventana en ambos sentidos, lo cual en pronóstico es discutible dado que el futuro no se lee al revés, pero dentro de una ventana cerrada puede caracterizar mejor el ciclo. Las unidades controlan cuánto puede recordar la red a la vez. |


In [ ]:
# Grid search sobre las 24 configuraciones, con validacion de origen rodante.
# El resultado se cachea en resultados/tablas/ para no repetir el tuneo en cada
# reejecucion; con usar_cache=False se fuerza a correrlo de nuevo desde cero.
grilla = utils.grilla_lstm()
print(f"Configuraciones a evaluar: {len(grilla)}")
print(f"Cortes de validacion     : {utils.cortes_origen_rodante(len(serie_train_log))}")
print(f"Semilla                  : {config.SEMILLA}")
print()

tuneo = utils.tunear_con_cache(SERIE, serie_train_log.values, grilla, usar_cache=True)

columnas = ["presentacion", "familia", "ventana", "unidades", "capas", "dropout",
            "RMSE_val", "MAE_val", "RMSE_val_sin_ultimo", "RMSE_std"]
tuneo[columnas].head(10).round(0)


In [ ]:
# Promedio por eje, para ver que decidio de verdad el resultado y que fue ruido.
print("RMSE de validacion promedio por presentacion:")
display(tuneo.groupby("presentacion")["RMSE_val"].agg(["mean", "min", "count"]).round(0))

print("RMSE de validacion promedio por familia de arquitectura:")
display(tuneo.groupby("familia")["RMSE_val"].agg(["mean", "min", "count"]).round(0))

print("RMSE de validacion promedio por ventana:")
display(tuneo.groupby("ventana")["RMSE_val"].agg(["mean", "min", "count"]).round(0))

# Configuracion ganadora del grid: la de menor RMSE promedio en los 4 cortes.
mejor = tuneo.iloc[0]
CFG_GANADORA = dict(
    presentacion=mejor["presentacion"], familia=mejor["familia"],
    ventana=int(mejor["ventana"]), unidades=int(mejor["unidades"]),
    capas=int(mejor["capas"]), dropout=float(mejor["dropout"]),
    bidireccional=bool(mejor["familia"] == "C_bidireccional"),
)
print("Configuracion ganadora del grid:")
for k, v in CFG_GANADORA.items():
    print(f"  {k:15s} {v}")
print(f"  RMSE_val         {mejor['RMSE_val']:,.0f}   (sin el corte pandemico: {mejor['RMSE_val_sin_ultimo']:,.0f})")
print(f"  RMSE por corte   {mejor['rmse_por_corte']}")


### **Los dos modelos**

A partir de la configuración ganadora del grid se instancian los dos modelos que pide el enunciado.
Ambos comparten la presentación, la ventana, la arquitectura y el número de unidades, y difieren
únicamente en la **tasa de aprendizaje**, de tal forma que la comparación aísla el efecto de ese
parámetro en lugar de mezclar varios cambios a la vez:

- **Modelo 1** — tasa de aprendizaje de **0.001**, que es la configuración con la que se recorrió la grilla.
- **Modelo 2** — tasa de aprendizaje de **0.01**, es decir pasos de corrección diez veces más grandes.


In [ ]:
# Los dos modelos: misma configuracion pero distinta tasa de aprendizaje.
SEMILLAS = [config.SEMILLA, config.SEMILLA + 1, config.SEMILLA + 2]
H = len(serie_prueba_log)

modelos, pronosticos, filas = {}, {}, []
for nombre, lr in utils.LR_MODELOS.items():
    cfg = {**CFG_GANADORA, "tasa_aprendizaje": lr}
    corridas = []
    for semilla in SEMILLAS:
        ajuste = utils.ajustar_lstm(serie_train_log.values, cfg, semilla=semilla)
        pred_log = utils.pronosticar_log(ajuste, H)
        m = utils.metricas(serie_prueba_log.values, pred_log)
        corridas.append({"semilla": semilla, "pred": pred_log, "ajuste": ajuste, **m})

    # Se reporta la corrida de la semilla base y el promedio entre semillas.
    base = corridas[0]
    modelos[nombre] = base["ajuste"]
    pronosticos[nombre] = pd.Series(base["pred"], index=serie_prueba_log.index)
    filas.append({
        "modelo": nombre,
        "tasa_aprendizaje": lr,
        "MAE": base["MAE"], "RMSE": base["RMSE"], "MAPE_%": base["MAPE_%"],
        "RMSE_prom_3_semillas": np.mean([c["RMSE"] for c in corridas]),
        "RMSE_desv_semillas": np.std([c["RMSE"] for c in corridas]),
        "epocas": base["ajuste"]["epocas_corridas"],
        "parametros": base["ajuste"]["modelo"].count_params(),
    })

comparacion_lstm = pd.DataFrame(filas).set_index("modelo")
display(comparacion_lstm.round(0))

modelos["Modelo 1"]["modelo"].summary()


In [ ]:
# Real vs los dos modelos sobre el conjunto de prueba, en escala de viajeros.
real = np.exp(serie_prueba_log)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.exp(serie_train_log).index, np.exp(serie_train_log).values,
        color="gray", linewidth=0.9, label="entrenamiento")
ax.plot(real.index, real.values, color="black", linewidth=2.5, label="real (prueba)")
for nombre, pred in pronosticos.items():
    lr = utils.LR_MODELOS[nombre]
    ax.plot(pred.index, np.exp(pred.values), marker=".", alpha=0.85,
            label=f"{nombre} (lr={lr})")
ax.axvline(serie_prueba_log.index[0], color="gray", linestyle="--", linewidth=1)
ax.set_title(f"LSTM: pronostico recursivo a 63 meses vs real — {config.SERIES_LAB1[SERIE]['etiqueta']}")
ax.set_ylabel("Numero de viajeros")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()

# Curvas de perdida: sirven para verificar que el entrenamiento converge y que
# la tasa mas alta no queda oscilando sin estabilizarse.
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for eje, (nombre, ajuste) in zip(ax, modelos.items()):
    eje.plot(ajuste["historia"].history["loss"], linewidth=1)
    eje.set_title(f"{nombre} (lr={utils.LR_MODELOS[nombre]}) — perdida MSE")
    eje.set_xlabel("epoca")
    eje.set_yscale("log")
plt.tight_layout()
plt.show()
